# NEU-CLS: Where robust defect classification degrades

**New reproducible Random Forest benchmark — not the missing 100% ResNet Kaggle notebook.**

Official NEU-CLS corpus: 1,800 images, six classes, [Figshare DOI 10.6084/m9.figshare.28903550.v1](https://doi.org/10.6084/m9.figshare.28903550.v1). This notebook reproduces the **new, source-verified classical CV experiment** in the NEXUS Visual QC repo. It does not claim to recover the historical Kaggle CNN notebook or validate its 100% result.

**Kaggle instructions:** attach the legitimate NEU-CLS dataset, preserving `train/train/images` and `valid/valid/images` folder structure (or run inside the cloned repository's `analysis/.cache/NEU-CLS` root). The notebook NEVER uploads the dataset or requires any Kaggle API token.

All transforms are applied only to the **same held-out images**. No training samples are perturbed, and the Random Forest is not refitted after a perturbation. Interpret metric changes as evidence for this one balanced benchmark, not production-line reliability.

In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
CLASSES = ("crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches")
EXPECTED_HOLDOUT_SHA256 = "26f63a336a40856b4af09a6ec8bf0daa27e6801ab79cb0cd8a29d253ab9480f5"
MODES = ("clean", "gaussian_noise_sigma_12", "gaussian_blur_5", "brightness_plus_25", "contrast_065", "jpeg_quality_45")


In [ ]:
roots = [Path("/kaggle/input"), Path("analysis/.cache/NEU-CLS")]
candidates = []
for root in roots:
    if root.exists():
        for train_dir in root.rglob("train/train/images"):
            valid_dir = train_dir.parents[2] / "valid" / "valid" / "images"
            if valid_dir.is_dir():
                candidates.append((train_dir, valid_dir))
assert len(candidates) == 1, (
    "Expected exactly one NEU-CLS dataset with train/train/images and valid/valid/images; "
    "attach/extract the official Figshare dataset before running."
)
train_dir, valid_dir = candidates[0]
paths = [*sorted(train_dir.glob("*.jpg")), *sorted(valid_dir.glob("*.jpg"))]
labels = np.array([p.stem.rsplit("_", 1)[0] for p in paths])
assert len(paths) == 1800 and Counter(labels) == {name: 300 for name in CLASSES}
train_idx, test_idx = train_test_split(
    np.arange(len(paths)), test_size=0.20, random_state=SEED, stratify=labels
)
holdout_sha = hashlib.sha256(
    "\n".join(paths[i].name for i in test_idx).encode("utf-8")
).hexdigest()
assert holdout_sha == EXPECTED_HOLDOUT_SHA256, (
    "Holdout identity differs from published NEXUS evidence. Do not compare metrics."
)
print(f"Verified 1,800 images; 1,440 train / 360 held out. Holdout SHA: {holdout_sha}")


In [ ]:
HOG = cv2.HOGDescriptor((64,64),(16,16),(8,8),(8,8),9)

def gabor_features(image64):
    out=[]
    for theta_deg in (0,45,90,135):
        theta=np.deg2rad(theta_deg)
        for lam in (4.0,8.0):
            kernel=cv2.getGaborKernel((15,15),sigma=4.0,theta=theta,lambd=lam,gamma=0.5,psi=0,ktype=cv2.CV_32F)
            resp=cv2.filter2D(image64.astype(np.float32),cv2.CV_32F,kernel)
            out.extend([float(resp.mean()),float(resp.std())])
    return np.array(out,dtype=np.float32)

def grid_features(image64):
    out=[]
    h,w=image64.shape
    ch,cw=h//4,w//4
    for i in range(4):
        for j in range(4):
            cell=image64[i*ch:(i+1)*ch,j*cw:(j+1)*cw].astype(np.float32)
            out.extend([float(cell.mean()),float(cell.std())])
    return np.array(out,dtype=np.float32)

def get_features(image):
    resized=cv2.resize(image,(64,64),interpolation=cv2.INTER_AREA)
    return np.concatenate([HOG.compute(resized).reshape(-1),gabor_features(resized),grid_features(resized)])

def perturb(image, mode, seed):
    image=np.asarray(image,dtype=np.uint8)
    if mode=="clean": return image.copy()
    if mode=="gaussian_noise_sigma_12":
        noise=np.random.default_rng(seed).normal(0,12,image.shape)
        return np.clip(image.astype(np.float32)+noise,0,255).astype(np.uint8)
    if mode=="gaussian_blur_5": return cv2.GaussianBlur(image,(5,5),0)
    if mode=="brightness_plus_25": return np.clip(image.astype(np.int16)+25,0,255).astype(np.uint8)
    if mode=="contrast_065":
        return np.clip((image.astype(np.float32)-127.5)*0.65+127.5,0,255).astype(np.uint8)
    if mode=="jpeg_quality_45":
        ok, encoded=cv2.imencode(".jpg",image,[cv2.IMWRITE_JPEG_QUALITY,45])
        assert ok
        result=cv2.imdecode(encoded,cv2.IMREAD_GRAYSCALE)
        assert result is not None and result.shape==image.shape
        return result
    raise ValueError("Unknown mode "+mode)


In [ ]:
def load_gray(path):
    image=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if image is None: raise ValueError("Unreadable image "+str(path))
    return image

x_train=np.vstack([get_features(load_gray(paths[i])) for i in train_idx])
model=RandomForestClassifier(
    n_estimators=250, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"
)
model.fit(x_train,labels[train_idx])
holdout_images=[load_gray(paths[i]) for i in test_idx]
y_test=labels[test_idx]
results={}
clean_pred=None
for mode in MODES:
    x_test=np.vstack([
        get_features(perturb(img,mode,SEED+int(i)))
        for img,i in zip(holdout_images,test_idx,strict=True)
    ])
    prediction=model.predict(x_test)
    if clean_pred is None: clean_pred=prediction.copy()
    results[mode]={
        "samples":len(y_test),
        "accuracy":round(float(accuracy_score(y_test,prediction)),6),
        "macro_f1":round(float(f1_score(y_test,prediction,average="macro",zero_division=0)),6),
        "flip_fraction_vs_clean":round(float(np.mean(prediction!=clean_pred)),6),
    }
for mode in MODES:
    results[mode]["accuracy_delta_percentage_points"]=round(
        100*(results[mode]["accuracy"]-results["clean"]["accuracy"]),4
    )
print(json.dumps(results,indent=2))


In [ ]:
out=Path("/kaggle/working/neu_cls_robustness_results.json") if Path("/kaggle/working").exists() else Path("neu_cls_robustness_results.json")
out.write_text(json.dumps({
    "study":"new NEU-CLS random-forest robustness reproduction, not original Kaggle CNN notebook",
    "dataset_doi":"10.6084/m9.figshare.28903550.v1",
    "holdout_image_list_sha256":holdout_sha,
    "seed":SEED,
    "model":"250-tree RF, HOG+Gabor+grid",
    "opencv_version":cv2.__version__,
    "scikit_learn_version":sklearn.__version__,
    "results":results,
},indent=2)+"\n",encoding="utf-8")
print("Saved",out)
